## Module 1: Crew Duty Intelligence System

### Business Problem

Airlines don't just worry about delays.

A delayed flight can cause:

- Crew exceeding legal duty hours
- Crew fatigue
- Flight cancellations
- Crew replacements
- Passenger compensation
- Schedule disruptions

> Our system will predict these operational risks before they happen.

#### Setup

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window

#### Load Silver & Gold Tables

In [0]:
# ==================================================
# Load Silver Tables
# ==================================================

flight_fact = spark.table("aviation_silver.flight_fact")

delay_fact = spark.table("aviation_silver.delay_fact")

# ==================================================
# Load Gold Tables
# ==================================================

gold_route = spark.table("aviation_gold.route_performance")

gold_airport = spark.table("aviation_gold.airport_performance")

gold_time = spark.table("aviation_gold.time_bucket_performance")

#### Build the Crew Base Table

In [0]:
from pyspark.sql.functions import *

crew_base_df = (
    flight_fact.alias("f")
    .join(delay_fact.alias("d"), "flight_id", "left")
    .select(
        "flight_id",
        "FL_DATE",
        "AIRLINE_CODE",
        "FL_NUMBER",
        "ORIGIN_AIRPORT_CODE",
        "DEST_AIRPORT_CODE",
        "CRS_DEP_TIME",
        "DEP_TIME",
        "CRS_ARR_TIME",
        "ARR_TIME",
        "DISTANCE",
        "DISTANCE_BAND",
        "DEPARTURE_TIME_BUCKET",
        "DEP_DELAY",
        "ARR_DELAY",
        "TOTAL_DELAY",
        "DOMINANT_DELAY_REASON",
    )
)

In [0]:
print("=" * 100)
print("Crew Base Records :", crew_base_df.count())
print("=" * 100)
crew_base_df.printSchema()
crew_base_df.show(10, truncate=False)

Crew Base Records : 2000000
root
 |-- flight_id: long (nullable = true)
 |-- FL_DATE: date (nullable = true)
 |-- AIRLINE_CODE: string (nullable = true)
 |-- FL_NUMBER: double (nullable = true)
 |-- ORIGIN_AIRPORT_CODE: string (nullable = true)
 |-- DEST_AIRPORT_CODE: string (nullable = true)
 |-- CRS_DEP_TIME: integer (nullable = true)
 |-- DEP_TIME: double (nullable = true)
 |-- CRS_ARR_TIME: double (nullable = true)
 |-- ARR_TIME: double (nullable = true)
 |-- DISTANCE: double (nullable = true)
 |-- DISTANCE_BAND: string (nullable = true)
 |-- DEPARTURE_TIME_BUCKET: string (nullable = true)
 |-- DEP_DELAY: double (nullable = true)
 |-- ARR_DELAY: double (nullable = true)
 |-- TOTAL_DELAY: double (nullable = true)
 |-- DOMINANT_DELAY_REASON: string (nullable = true)

+-----------+----------+------------+---------+-------------------+-----------------+------------+--------+------------+--------+--------+-------------+---------------------+---------+---------+-----------+--------------

#### Sequence Flights Within Airline + Date

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import *

# ==================================================
# Flight sequence within Airline + Date
# ==================================================

flight_window = Window.partitionBy("AIRLINE_CODE", "FL_DATE").orderBy("CRS_DEP_TIME")

crew_base_df = crew_base_df.withColumn(
    "FLIGHT_SEQUENCE", row_number().over(flight_window)
)

In [0]:
crew_base_df = crew_base_df.withColumn(
    "AIRCRAFT_NUMBER", floor((col("FLIGHT_SEQUENCE") - 1) / 5) + 1
).withColumn(
    "AIRCRAFT_ID",
    concat(col("AIRLINE_CODE"), lit("_AC_"), lpad(col("AIRCRAFT_NUMBER"), 4, "0")),
)

In [0]:
crew_base_df = crew_base_df.withColumn(
    "CREW_ID", regexp_replace(col("AIRCRAFT_ID"), "_AC_", "_CR_")
)

In [0]:
crew_base_df.select(
    "FL_DATE",
    "AIRLINE_CODE",
    "CRS_DEP_TIME",
    "FLIGHT_SEQUENCE",
    "AIRCRAFT_ID",
    "CREW_ID",
).show(30, truncate=False)

+----------+------------+------------+---------------+-----------+----------+
|FL_DATE   |AIRLINE_CODE|CRS_DEP_TIME|FLIGHT_SEQUENCE|AIRCRAFT_ID|CREW_ID   |
+----------+------------+------------+---------------+-----------+----------+
|2019-01-04|9E          |653         |1              |9E_AC_0001 |9E_CR_0001|
|2019-01-04|9E          |725         |2              |9E_AC_0001 |9E_CR_0001|
|2019-01-04|9E          |840         |3              |9E_AC_0001 |9E_CR_0001|
|2019-01-04|9E          |844         |4              |9E_AC_0001 |9E_CR_0001|
|2019-01-04|9E          |850         |5              |9E_AC_0001 |9E_CR_0001|
|2019-01-04|9E          |855         |6              |9E_AC_0002 |9E_CR_0002|
|2019-01-04|9E          |1005        |7              |9E_AC_0002 |9E_CR_0002|
|2019-01-04|9E          |1007        |8              |9E_AC_0002 |9E_CR_0002|
|2019-01-04|9E          |1117        |9              |9E_AC_0002 |9E_CR_0002|
|2019-01-04|9E          |1200        |10             |9E_AC_0002

#### Assumptions

Average cruise speed = 500 mph

Flight time (minutes) = (DISTANCE / 500) `×` 60

In [0]:
from pyspark.sql.functions import *

crew_base_df = crew_base_df.withColumn(
    "ESTIMATED_FLIGHT_TIME_MIN", round((col("DISTANCE") / 500) * 60, 0)
)
print("=" * 100)
print("Estimated Flight Time Added")
print("=" * 100)
crew_base_df.select("DISTANCE", "DISTANCE_BAND", "ESTIMATED_FLIGHT_TIME_MIN").show(
    10, truncate=False
)

Estimated Flight Time Added
+--------+-------------+-------------------------+
|DISTANCE|DISTANCE_BAND|ESTIMATED_FLIGHT_TIME_MIN|
+--------+-------------+-------------------------+
|901.0   |MEDIUM_HAUL  |108.0                    |
|365.0   |SHORT_HAUL   |44.0                     |
|1046.0  |MEDIUM_HAUL  |126.0                    |
|399.0   |SHORT_HAUL   |48.0                     |
|280.0   |SHORT_HAUL   |34.0                     |
|700.0   |MEDIUM_HAUL  |84.0                     |
|305.0   |SHORT_HAUL   |37.0                     |
|216.0   |SHORT_HAUL   |26.0                     |
|402.0   |SHORT_HAUL   |48.0                     |
|1106.0  |MEDIUM_HAUL  |133.0                    |
+--------+-------------+-------------------------+
only showing top 10 rows


#### Estimate Crew Duty Time

In [0]:
from pyspark.sql.functions import *

crew_base_df = crew_base_df.withColumn(
    "POSITIVE_DELAY", least(greatest(col("TOTAL_DELAY"), lit(0)), lit(300))
).withColumn(
    "ESTIMATED_DUTY_TIME_MIN",
    col("ESTIMATED_FLIGHT_TIME_MIN") + lit(45) + lit(45) + col("POSITIVE_DELAY"),
)

In [0]:
crew_base_df.select(
    round(avg("ESTIMATED_DUTY_TIME_MIN"), 2).alias("AVG_DUTY_TIME"),
    min("ESTIMATED_DUTY_TIME_MIN").alias("MIN_DUTY_TIME"),
    max("ESTIMATED_DUTY_TIME_MIN").alias("MAX_DUTY_TIME"),
).show()

+-------------+-------------+-------------+
|AVG_DUTY_TIME|MIN_DUTY_TIME|MAX_DUTY_TIME|
+-------------+-------------+-------------+
|       198.34|         93.0|        988.0|
+-------------+-------------+-------------+



#### Convert Scheduled Times to Minutes

In [0]:
from pyspark.sql.functions import *

# ==================================================
# Convert HHMM to Minutes Since Midnight
# ==================================================

crew_base_df = crew_base_df.withColumn(
    "CRS_DEP_MINUTES",
    (floor(col("CRS_DEP_TIME") / 100) * 60) + (col("CRS_DEP_TIME") % 100),
).withColumn(
    "CRS_ARR_MINUTES",
    (floor(col("CRS_ARR_TIME") / 100) * 60) + (col("CRS_ARR_TIME") % 100),
)

print("=" * 100)
print("Scheduled Time Converted To Minutes")
print("=" * 100)

crew_base_df.select(
    "CRS_DEP_TIME", "CRS_DEP_MINUTES", "CRS_ARR_TIME", "CRS_ARR_MINUTES"
).show(10, truncate=False)

Scheduled Time Converted To Minutes
+------------+---------------+------------+---------------+
|CRS_DEP_TIME|CRS_DEP_MINUTES|CRS_ARR_TIME|CRS_ARR_MINUTES|
+------------+---------------+------------+---------------+
|1246        |766            |1618.0      |978.0          |
|1115        |675            |1235.0      |755.0          |
|759         |479            |932.0       |572.0          |
|1158        |718            |1430.0      |870.0          |
|1713        |1033           |1922.0      |1162.0         |
|500         |300            |550.0       |350.0          |
|1608        |968            |1738.0      |1058.0         |
|620         |380            |739.0       |459.0          |
|1645        |1005           |2017.0      |1217.0         |
|835         |515            |955.0       |595.0          |
+------------+---------------+------------+---------------+
only showing top 10 rows


In [0]:
from pyspark.sql.functions import *

crew_base_df = crew_base_df.withColumn(
    "CRS_ARR_MINUTES",
    when(
        col("CRS_ARR_MINUTES") < col("CRS_DEP_MINUTES"), col("CRS_ARR_MINUTES") + 1440
    ).otherwise(col("CRS_ARR_MINUTES")),
)
print("=" * 100)
print("Overnight Flights Corrected")
print("=" * 100)
crew_base_df.select(
    "CRS_DEP_TIME", "CRS_ARR_TIME", "CRS_DEP_MINUTES", "CRS_ARR_MINUTES"
).show(10, truncate=False)

Overnight Flights Corrected
+------------+------------+---------------+---------------+
|CRS_DEP_TIME|CRS_ARR_TIME|CRS_DEP_MINUTES|CRS_ARR_MINUTES|
+------------+------------+---------------+---------------+
|1246        |1618.0      |766            |978.0          |
|1115        |1235.0      |675            |755.0          |
|759         |932.0       |479            |572.0          |
|1158        |1430.0      |718            |870.0          |
|1713        |1922.0      |1033           |1162.0         |
|500         |550.0       |300            |350.0          |
|1608        |1738.0      |968            |1058.0         |
|620         |739.0       |380            |459.0          |
|1645        |2017.0      |1005           |1217.0         |
|835         |955.0       |515            |595.0          |
+------------+------------+---------------+---------------+
only showing top 10 rows


#### Build the Simulation Dataset (v1)

In [0]:
simulation_df = crew_base_df.select(
    "flight_id",
    "FL_DATE",
    "AIRLINE_CODE",
    "FL_NUMBER",
    "ORIGIN_AIRPORT_CODE",
    "DEST_AIRPORT_CODE",
    "CRS_DEP_TIME",
    "CRS_ARR_TIME",
    "CRS_DEP_MINUTES",
    "CRS_ARR_MINUTES",
    "DISTANCE",
    "ESTIMATED_FLIGHT_TIME_MIN",
    "ESTIMATED_DUTY_TIME_MIN",
    "TOTAL_DELAY",
    "DOMINANT_DELAY_REASON",
)
print("=" * 100)
print("Simulation Dataset Ready")
print("=" * 100)
simulation_df.show(10, truncate=False)

Simulation Dataset Ready
+-----------+----------+------------+---------+-------------------+-----------------+------------+------------+---------------+---------------+--------+-------------------------+-----------------------+-----------+---------------------+
|flight_id  |FL_DATE   |AIRLINE_CODE|FL_NUMBER|ORIGIN_AIRPORT_CODE|DEST_AIRPORT_CODE|CRS_DEP_TIME|CRS_ARR_TIME|CRS_DEP_MINUTES|CRS_ARR_MINUTES|DISTANCE|ESTIMATED_FLIGHT_TIME_MIN|ESTIMATED_DUTY_TIME_MIN|TOTAL_DELAY|DOMINANT_DELAY_REASON|
+-----------+----------+------------+---------+-------------------+-----------------+------------+------------+---------------+---------------+--------+-------------------------+-----------------------+-----------+---------------------+
|34359738377|2023-02-24|AA          |1194.0   |DFW                |ILM              |1246        |1618.0      |766            |978.0          |1106.0  |133.0                    |242.0                  |19.0       |NAS                  |
|34359738376|2022-10-22|WN 

#### Define the Simulation Output Schema (v1)

In [0]:
from pyspark.sql.types import *

simulation_schema = StructType(
    [
        StructField("flight_id", LongType()),
        StructField("FL_DATE", DateType()),
        StructField("AIRLINE_CODE", StringType()),
        StructField("FL_NUMBER", DoubleType()),
        StructField("ORIGIN_AIRPORT_CODE", StringType()),
        StructField("DEST_AIRPORT_CODE", StringType()),
        StructField("CRS_DEP_TIME", IntegerType()),
        StructField("CRS_ARR_TIME", DoubleType()),
        StructField("CRS_DEP_MINUTES", IntegerType()),
        StructField("CRS_ARR_MINUTES", DoubleType()),
        StructField("DISTANCE", DoubleType()),
        StructField("ESTIMATED_FLIGHT_TIME_MIN", DoubleType()),
        StructField("ESTIMATED_DUTY_TIME_MIN", DoubleType()),
        StructField("TOTAL_DELAY", DoubleType()),
        StructField("DOMINANT_DELAY_REASON", StringType()),
        StructField("AIRCRAFT_ID", StringType()),
        StructField("CREW_ID", StringType()),
        StructField("FLIGHT_LEG_NUMBER", IntegerType()),
        StructField("AIRCRAFT_UTILIZATION_MIN", DoubleType()),
    ]
)

#### Imports for the Pandas UDF Simulation

In [0]:
import pandas as pd
import numpy as np

#### Aircraft Scheduling Engine (Pandas UDF)

In [0]:
import pandas as pd

# ============================================================
# Aircraft Scheduling Engine
# ============================================================

MAX_DUTY_MINUTES = 660  # 11 Hours
TURNAROUND_TIME = 45  # Minutes


def simulate_aircraft_assignment(pdf: pd.DataFrame) -> pd.DataFrame:

    # --------------------------------------------------------
    # Sort Flights
    # --------------------------------------------------------

    pdf = pdf.sort_values("CRS_DEP_MINUTES").reset_index(drop=True)

    airline = pdf.iloc[0]["AIRLINE_CODE"]

    # --------------------------------------------------------
    # Fleet Dictionary
    # --------------------------------------------------------

    fleet = {}

    aircraft_counter = 1

    aircraft_ids = []
    crew_ids = []
    leg_numbers = []
    utilization_minutes = []

    # --------------------------------------------------------
    # Process Every Flight
    # --------------------------------------------------------

    for _, flight in pdf.iterrows():

        dep_time = flight["CRS_DEP_MINUTES"]

        arr_time = flight["CRS_ARR_MINUTES"]

        duty = flight["ESTIMATED_DUTY_TIME_MIN"]

        assigned_aircraft = None

        # ----------------------------------------------------
        # Try Existing Aircraft
        # ----------------------------------------------------

        for aircraft_id, info in fleet.items():

            enough_capacity = info["utilization"] + duty <= MAX_DUTY_MINUTES

            available = info["available_time"] <= dep_time

            if enough_capacity and available:

                assigned_aircraft = aircraft_id

                break

        # ----------------------------------------------------
        # Create New Aircraft
        # ----------------------------------------------------

        if assigned_aircraft is None:

            assigned_aircraft = f"{airline}_AC_{aircraft_counter:04d}"

            fleet[assigned_aircraft] = {
                "utilization": 0,
                "available_time": 0,
                "legs": 0,
            }

            aircraft_counter += 1

        # ----------------------------------------------------
        # Update Aircraft State
        # ----------------------------------------------------

        fleet[assigned_aircraft]["utilization"] += duty

        fleet[assigned_aircraft]["available_time"] = arr_time + TURNAROUND_TIME

        fleet[assigned_aircraft]["legs"] += 1

        aircraft_ids.append(assigned_aircraft)

        crew_ids.append(assigned_aircraft.replace("_AC_", "_CR_"))

        leg_numbers.append(fleet[assigned_aircraft]["legs"])

        utilization_minutes.append(fleet[assigned_aircraft]["utilization"])

    # --------------------------------------------------------
    # Output Columns
    # --------------------------------------------------------

    pdf["AIRCRAFT_ID"] = aircraft_ids

    pdf["CREW_ID"] = crew_ids

    pdf["FLIGHT_LEG_NUMBER"] = leg_numbers

    pdf["AIRCRAFT_UTILIZATION_MIN"] = utilization_minutes

    return pdf

#### Run the Aircraft Assignment Simulation (v1)

In [0]:
crew_simulation_df = simulation_df.groupBy("AIRLINE_CODE", "FL_DATE").applyInPandas(
    simulate_aircraft_assignment, schema=simulation_schema
)

In [0]:
print("=" * 100)
print("Crew Simulation Records :", crew_simulation_df.count())
print("=" * 100)
crew_simulation_df.select(
    "FL_DATE",
    "AIRLINE_CODE",
    "CRS_DEP_TIME",
    "AIRCRAFT_ID",
    "CREW_ID",
    "FLIGHT_LEG_NUMBER",
    "AIRCRAFT_UTILIZATION_MIN",
).orderBy("FL_DATE", "AIRLINE_CODE", "CRS_DEP_TIME").show(50, truncate=False)

Crew Simulation Records : 2000000
+----------+------------+------------+-----------+----------+-----------------+------------------------+
|FL_DATE   |AIRLINE_CODE|CRS_DEP_TIME|AIRCRAFT_ID|CREW_ID   |FLIGHT_LEG_NUMBER|AIRCRAFT_UTILIZATION_MIN|
+----------+------------+------------+-----------+----------+-----------------+------------------------+
|2019-01-01|9E          |600         |9E_AC_0001 |9E_CR_0001|1                |158.0                   |
|2019-01-01|9E          |730         |9E_AC_0002 |9E_CR_0002|1                |281.0                   |
|2019-01-01|9E          |805         |9E_AC_0003 |9E_CR_0003|1                |107.0                   |
|2019-01-01|9E          |836         |9E_AC_0004 |9E_CR_0004|1                |171.0                   |
|2019-01-01|9E          |845         |9E_AC_0005 |9E_CR_0005|1                |124.0                   |
|2019-01-01|9E          |850         |9E_AC_0006 |9E_CR_0006|1                |143.0                   |
|2019-01-01|9E       

#### Refine Aircraft Occupied Time

In [0]:
from pyspark.sql.functions import *

TURNAROUND_TIME = 45
crew_base_df = crew_base_df.withColumn(
    "AIRCRAFT_OCCUPIED_MIN",
    (col("CRS_ARR_MINUTES") - col("CRS_DEP_MINUTES")) + lit(TURNAROUND_TIME),
)
crew_base_df.select(
    "CRS_DEP_TIME", "CRS_ARR_TIME", "AIRCRAFT_OCCUPIED_MIN", "ESTIMATED_DUTY_TIME_MIN"
).show(10, False)

+------------+------------+---------------------+-----------------------+
|CRS_DEP_TIME|CRS_ARR_TIME|AIRCRAFT_OCCUPIED_MIN|ESTIMATED_DUTY_TIME_MIN|
+------------+------------+---------------------+-----------------------+
|1246        |1618.0      |257.0                |242.0                  |
|1115        |1235.0      |125.0                |138.0                  |
|759         |932.0       |138.0                |127.0                  |
|1158        |1430.0      |197.0                |138.0                  |
|1713        |1922.0      |174.0                |174.0                  |
|500         |550.0       |95.0                 |116.0                  |
|1608        |1738.0      |135.0                |166.0                  |
|620         |739.0       |124.0                |134.0                  |
|1645        |2017.0      |257.0                |216.0                  |
|835         |955.0       |125.0                |131.0                  |
+------------+------------+-----------

In [0]:
from pyspark.sql.functions import *

TURNAROUND_TIME = 45
crew_base_df = crew_base_df.withColumn(
    "AIRCRAFT_OCCUPIED_MIN",
    (col("CRS_ARR_MINUTES") - col("CRS_DEP_MINUTES")) + lit(TURNAROUND_TIME),
)
print("=" * 100)
print("Aircraft Occupied Time Added")
print("=" * 100)
crew_base_df.select(
    "CRS_DEP_TIME", "CRS_ARR_TIME", "AIRCRAFT_OCCUPIED_MIN", "ESTIMATED_DUTY_TIME_MIN"
).show(10, truncate=False)

Aircraft Occupied Time Added
+------------+------------+---------------------+-----------------------+
|CRS_DEP_TIME|CRS_ARR_TIME|AIRCRAFT_OCCUPIED_MIN|ESTIMATED_DUTY_TIME_MIN|
+------------+------------+---------------------+-----------------------+
|1246        |1618.0      |257.0                |242.0                  |
|1115        |1235.0      |125.0                |138.0                  |
|759         |932.0       |138.0                |127.0                  |
|1158        |1430.0      |197.0                |138.0                  |
|1713        |1922.0      |174.0                |174.0                  |
|500         |550.0       |95.0                 |116.0                  |
|1608        |1738.0      |135.0                |166.0                  |
|620         |739.0       |124.0                |134.0                  |
|1645        |2017.0      |257.0                |216.0                  |
|835         |955.0       |125.0                |131.0                  |
+--------

#### Rebuild the Simulation Dataset (v2)

In [0]:
simulation_df = crew_base_df.select(
    "flight_id",
    "FL_DATE",
    "AIRLINE_CODE",
    "FL_NUMBER",
    "ORIGIN_AIRPORT_CODE",
    "DEST_AIRPORT_CODE",
    "CRS_DEP_TIME",
    "CRS_ARR_TIME",
    "CRS_DEP_MINUTES",
    "CRS_ARR_MINUTES",
    "DISTANCE",
    "ESTIMATED_FLIGHT_TIME_MIN",
    "ESTIMATED_DUTY_TIME_MIN",
    "AIRCRAFT_OCCUPIED_MIN",
    "TOTAL_DELAY",
    "DOMINANT_DELAY_REASON",
)
print("=" * 100)
print("Simulation Dataset Ready")
print("=" * 100)
simulation_df.show(10, truncate=False)

Simulation Dataset Ready
+-----------+----------+------------+---------+-------------------+-----------------+------------+------------+---------------+---------------+--------+-------------------------+-----------------------+---------------------+-----------+---------------------+
|flight_id  |FL_DATE   |AIRLINE_CODE|FL_NUMBER|ORIGIN_AIRPORT_CODE|DEST_AIRPORT_CODE|CRS_DEP_TIME|CRS_ARR_TIME|CRS_DEP_MINUTES|CRS_ARR_MINUTES|DISTANCE|ESTIMATED_FLIGHT_TIME_MIN|ESTIMATED_DUTY_TIME_MIN|AIRCRAFT_OCCUPIED_MIN|TOTAL_DELAY|DOMINANT_DELAY_REASON|
+-----------+----------+------------+---------+-------------------+-----------------+------------+------------+---------------+---------------+--------+-------------------------+-----------------------+---------------------+-----------+---------------------+
|34359738377|2023-02-24|AA          |1194.0   |DFW                |ILM              |1246        |1618.0      |766            |978.0          |1106.0  |133.0                    |242.0               

#### Redefine the Simulation Output Schema (v2)

In [0]:
from pyspark.sql.types import *

simulation_schema = StructType(
    [
        StructField("flight_id", LongType()),
        StructField("FL_DATE", DateType()),
        StructField("AIRLINE_CODE", StringType()),
        StructField("FL_NUMBER", DoubleType()),
        StructField("ORIGIN_AIRPORT_CODE", StringType()),
        StructField("DEST_AIRPORT_CODE", StringType()),
        StructField("CRS_DEP_TIME", IntegerType()),
        StructField("CRS_ARR_TIME", DoubleType()),
        StructField("CRS_DEP_MINUTES", IntegerType()),
        StructField("CRS_ARR_MINUTES", DoubleType()),
        StructField("DISTANCE", DoubleType()),
        StructField("ESTIMATED_FLIGHT_TIME_MIN", DoubleType()),
        StructField("ESTIMATED_DUTY_TIME_MIN", DoubleType()),
        StructField("AIRCRAFT_OCCUPIED_MIN", DoubleType()),
        StructField("TOTAL_DELAY", DoubleType()),
        StructField("DOMINANT_DELAY_REASON", StringType()),
        StructField("AIRCRAFT_ID", StringType()),
        StructField("CREW_ID", StringType()),
        StructField("FLIGHT_LEG_NUMBER", IntegerType()),
        StructField("AIRCRAFT_UTILIZATION_MIN", DoubleType()),
    ]
)

#### Aircraft Scheduling Engine (Updated, Load-Balanced)

In [0]:
import pandas as pd

MAX_AIRCRAFT_UTILIZATION = 660
TURNAROUND_TIME = 45


def simulate_aircraft_assignment(pdf: pd.DataFrame):

    pdf = pdf.sort_values("CRS_DEP_MINUTES").reset_index(drop=True)

    airline = pdf.iloc[0]["AIRLINE_CODE"]

    fleet = {}

    aircraft_counter = 1

    aircraft_ids = []
    crew_ids = []
    leg_numbers = []
    utilization = []

    for _, flight in pdf.iterrows():

        dep = flight["CRS_DEP_MINUTES"]

        arr = flight["CRS_ARR_MINUTES"]

        occupied = flight["AIRCRAFT_OCCUPIED_MIN"]

        best_aircraft = None

        highest_utilization = -1

        # ---------------------------------------------------
        # Find Best Existing Aircraft
        # ---------------------------------------------------

        for aircraft_id, info in fleet.items():

            enough_capacity = info["utilization"] + occupied <= MAX_AIRCRAFT_UTILIZATION

            available = info["available_time"] <= dep

            if enough_capacity and available:

                if info["utilization"] > highest_utilization:

                    highest_utilization = info["utilization"]

                    best_aircraft = aircraft_id

        # ---------------------------------------------------
        # Create New Aircraft
        # ---------------------------------------------------

        if best_aircraft is None:

            best_aircraft = f"{airline}_AC_{aircraft_counter:04d}"

            fleet[best_aircraft] = {"utilization": 0, "available_time": 0, "legs": 0}

            aircraft_counter += 1

        # ---------------------------------------------------
        # Update Aircraft
        # ---------------------------------------------------

        fleet[best_aircraft]["utilization"] += occupied

        fleet[best_aircraft]["available_time"] = arr + TURNAROUND_TIME

        fleet[best_aircraft]["legs"] += 1

        aircraft_ids.append(best_aircraft)

        crew_ids.append(best_aircraft.replace("_AC_", "_CR_"))

        leg_numbers.append(fleet[best_aircraft]["legs"])

        utilization.append(fleet[best_aircraft]["utilization"])

    pdf["AIRCRAFT_ID"] = aircraft_ids

    pdf["CREW_ID"] = crew_ids

    pdf["FLIGHT_LEG_NUMBER"] = leg_numbers

    pdf["AIRCRAFT_UTILIZATION_MIN"] = utilization

    return pdf

#### Run the Aircraft Assignment Simulation (v2)

In [0]:
crew_simulation_df = simulation_df.groupBy("AIRLINE_CODE", "FL_DATE").applyInPandas(
    simulate_aircraft_assignment, schema=simulation_schema
)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/group.py:303: UserWarning: Cannot infer the eval type from type hints.
  warnings.warn("Cannot infer the eval type from type hints.", UserWarning)


In [0]:
print("=" * 100)
print("Crew Simulation Records :", crew_simulation_df.count())
print("=" * 100)
crew_simulation_df.select(
    "FL_DATE",
    "AIRLINE_CODE",
    "CRS_DEP_TIME",
    "AIRCRAFT_ID",
    "CREW_ID",
    "FLIGHT_LEG_NUMBER",
    "AIRCRAFT_UTILIZATION_MIN",
    "AIRCRAFT_OCCUPIED_MIN",
).orderBy("FL_DATE", "AIRLINE_CODE", "CRS_DEP_TIME").show(100, truncate=False)

Crew Simulation Records : 2000000
+----------+------------+------------+-----------+----------+-----------------+------------------------+---------------------+
|FL_DATE   |AIRLINE_CODE|CRS_DEP_TIME|AIRCRAFT_ID|CREW_ID   |FLIGHT_LEG_NUMBER|AIRCRAFT_UTILIZATION_MIN|AIRCRAFT_OCCUPIED_MIN|
+----------+------------+------------+-----------+----------+-----------------+------------------------+---------------------+
|2019-01-01|9E          |600         |9E_AC_0001 |9E_CR_0001|1                |222.0                   |222.0                |
|2019-01-01|9E          |730         |9E_AC_0002 |9E_CR_0002|1                |211.0                   |211.0                |
|2019-01-01|9E          |805         |9E_AC_0003 |9E_CR_0003|1                |116.0                   |116.0                |
|2019-01-01|9E          |836         |9E_AC_0004 |9E_CR_0004|1                |118.0                   |118.0                |
|2019-01-01|9E          |845         |9E_AC_0005 |9E_CR_0005|1               

#### Crew Duty Assumptions

In [0]:
# ==================================================
# Duty Time Assumptions
# ==================================================

BRIEFING_TIME = 45
DEBRIEF_TIME = 30
DELAY_IMPACT_FACTOR = 0.50

#### Build the Crew Daily Summary

In [0]:
from pyspark.sql.functions import *

# ==================================================
# Crew Duty Assumptions
# ==================================================

BRIEFING_TIME = 45
DEBRIEF_TIME = 30
DELAY_IMPACT_FACTOR = 0.50

# ==================================================
# Crew Daily Summary
# ==================================================

crew_daily_summary = (
    crew_simulation_df.groupBy("FL_DATE", "AIRLINE_CODE", "CREW_ID", "AIRCRAFT_ID")
    .agg(
        # ==================================================
        # Crew Workload
        # ==================================================
        count("*").alias("TOTAL_FLIGHTS"),
        max("FLIGHT_LEG_NUMBER").alias("TOTAL_FLIGHT_LEGS"),
        # ==================================================
        # Duty Window
        # ==================================================
        min("CRS_DEP_MINUTES").alias("FIRST_DEPARTURE_MIN"),
        max("CRS_ARR_MINUTES").alias("LAST_ARRIVAL_MIN"),
        # ==================================================
        # Delay Metrics
        # ==================================================
        round(sum("TOTAL_DELAY"), 2).alias("TOTAL_DELAY_MIN"),
        round(avg("TOTAL_DELAY"), 2).alias("AVG_DELAY_PER_FLIGHT"),
        # ==================================================
        # Flight Metrics
        # ==================================================
        round(avg("ESTIMATED_FLIGHT_TIME_MIN"), 2).alias("AVG_FLIGHT_DURATION_MIN"),
        # ==================================================
        # Aircraft Utilization
        # ==================================================
        round(max("AIRCRAFT_UTILIZATION_MIN"), 2).alias("AIRCRAFT_UTILIZATION_MIN"),
    )
    # ==================================================
    # Duty Window
    # ==================================================
    .withColumn("DUTY_WINDOW_MIN", col("LAST_ARRIVAL_MIN") - col("FIRST_DEPARTURE_MIN"))
    # ==================================================
    # Effective Delay
    # ==================================================
    .withColumn(
        "EFFECTIVE_DELAY_MIN",
        round(col("TOTAL_DELAY_MIN") * lit(DELAY_IMPACT_FACTOR), 2),
    )
    # ==================================================
    # Duty Components
    # ==================================================
    .withColumn("BRIEFING_TIME_MIN", lit(BRIEFING_TIME))
    .withColumn("DEBRIEF_TIME_MIN", lit(DEBRIEF_TIME))
    # ==================================================
    # Final Duty Time
    # ==================================================
    .withColumn(
        "TOTAL_DUTY_MIN",
        col("DUTY_WINDOW_MIN")
        + col("BRIEFING_TIME_MIN")
        + col("DEBRIEF_TIME_MIN")
        + col("EFFECTIVE_DELAY_MIN"),
    )
    .withColumn("TOTAL_DUTY_HOURS", round(col("TOTAL_DUTY_MIN") / 60, 2))
    .withColumn(
        "AIRCRAFT_UTILIZATION_HOURS", round(col("AIRCRAFT_UTILIZATION_MIN") / 60, 2)
    )
)

print("=" * 100)
print("Crew Daily Summary Records :", crew_daily_summary.count())
print("=" * 100)

crew_daily_summary.show(10, truncate=False)

Crew Daily Summary Records : 748031
+----------+------------+----------+-----------+-------------+-----------------+-------------------+----------------+---------------+--------------------+-----------------------+------------------------+---------------+-------------------+-----------------+----------------+--------------+----------------+--------------------------+
|FL_DATE   |AIRLINE_CODE|CREW_ID   |AIRCRAFT_ID|TOTAL_FLIGHTS|TOTAL_FLIGHT_LEGS|FIRST_DEPARTURE_MIN|LAST_ARRIVAL_MIN|TOTAL_DELAY_MIN|AVG_DELAY_PER_FLIGHT|AVG_FLIGHT_DURATION_MIN|AIRCRAFT_UTILIZATION_MIN|DUTY_WINDOW_MIN|EFFECTIVE_DELAY_MIN|BRIEFING_TIME_MIN|DEBRIEF_TIME_MIN|TOTAL_DUTY_MIN|TOTAL_DUTY_HOURS|AIRCRAFT_UTILIZATION_HOURS|
+----------+------------+----------+-----------+-------------+-----------------+-------------------+----------------+---------------+--------------------+-----------------------+------------------------+---------------+-------------------+-----------------+----------------+--------------+-------

#### Refine Duty Window & Duty Time Calculation

In [0]:
crew_daily_summary = (
    crew_daily_summary.withColumn(
        "DUTY_WINDOW_MIN", col("LAST_ARRIVAL_MIN") - col("FIRST_DEPARTURE_MIN")
    )
    .withColumn(
        "EFFECTIVE_DELAY_MIN",
        round(col("TOTAL_DELAY_MIN") * lit(DELAY_IMPACT_FACTOR), 2),
    )
    .withColumn(
        "TOTAL_DUTY_MIN",
        col("DUTY_WINDOW_MIN")
        + lit(BRIEFING_TIME)
        + lit(DEBRIEF_TIME)
        + col("EFFECTIVE_DELAY_MIN"),
    )
    .withColumn("TOTAL_DUTY_HOURS", round(col("TOTAL_DUTY_MIN") / 60, 2))
    .withColumn(
        "AIRCRAFT_UTILIZATION_HOURS", round(col("AIRCRAFT_UTILIZATION_MIN") / 60, 2)
    )
)

In [0]:
from pyspark.sql.functions import *

crew_daily_summary = (
    crew_daily_summary.withColumn(
        "DUTY_WINDOW_MIN", (col("LAST_ARRIVAL_MIN") - col("FIRST_DEPARTURE_MIN"))
    )
    .withColumn(
        "TOTAL_DUTY_MIN",
        col("DUTY_WINDOW_MIN")
        + lit(BRIEFING_TIME)
        + lit(DEBRIEF_TIME)
        + col("TOTAL_DELAY_MIN"),
    )
    .withColumn("TOTAL_DUTY_HOURS", round(col("TOTAL_DUTY_MIN") / 60, 2))
    .withColumn(
        "AIRCRAFT_UTILIZATION_HOURS", round(col("AIRCRAFT_UTILIZATION_MIN") / 60, 2)
    )
)
print("=" * 100)
print("Duty Window Calculated")
print("=" * 100)
crew_daily_summary.select(
    "CREW_ID",
    "TOTAL_FLIGHTS",
    "FIRST_DEPARTURE_MIN",
    "LAST_ARRIVAL_MIN",
    "DUTY_WINDOW_MIN",
    "TOTAL_DELAY_MIN",
    "TOTAL_DUTY_HOURS",
).show(20, truncate=False)

Duty Window Calculated
+----------+-------------+-------------------+----------------+---------------+---------------+----------------+
|CREW_ID   |TOTAL_FLIGHTS|FIRST_DEPARTURE_MIN|LAST_ARRIVAL_MIN|DUTY_WINDOW_MIN|TOTAL_DELAY_MIN|TOTAL_DUTY_HOURS|
+----------+-------------+-------------------+----------------+---------------+---------------+----------------+
|9E_CR_0015|1            |491                |570.0           |79.0           |0.0            |2.57            |
|9E_CR_0007|5            |445                |1163.0          |718.0          |0.0            |13.22           |
|9E_CR_0014|1            |1162               |1305.0          |143.0          |53.0           |4.52            |
|9E_CR_0012|3            |725                |1398.0          |673.0          |122.0          |14.5            |
|9E_CR_0005|4            |491                |1158.0          |667.0          |0.0            |12.37           |
|9E_CR_0010|4            |616                |1330.0          |714.0     

In [0]:
crew_daily_summary.select(
    "FL_DATE",
    "CREW_ID",
    "AIRCRAFT_ID",
    "TOTAL_FLIGHTS",
    "TOTAL_FLIGHT_LEGS",
    "FIRST_DEPARTURE_MIN",
    "LAST_ARRIVAL_MIN",
    "DUTY_WINDOW_MIN",
    "TOTAL_DELAY_MIN",
    "TOTAL_DUTY_HOURS",
    "AIRCRAFT_UTILIZATION_HOURS",
).orderBy("FL_DATE", "CREW_ID").show(50, truncate=False)

+----------+----------+-----------+-------------+-----------------+-------------------+----------------+---------------+---------------+----------------+--------------------------+
|FL_DATE   |CREW_ID   |AIRCRAFT_ID|TOTAL_FLIGHTS|TOTAL_FLIGHT_LEGS|FIRST_DEPARTURE_MIN|LAST_ARRIVAL_MIN|DUTY_WINDOW_MIN|TOTAL_DELAY_MIN|TOTAL_DUTY_HOURS|AIRCRAFT_UTILIZATION_HOURS|
+----------+----------+-----------+-------------+-----------------+-------------------+----------------+---------------+---------------+----------------+--------------------------+
|2019-01-01|9E_CR_0001|9E_AC_0001 |3            |3                |360                |963.0           |603.0          |0.0            |11.3            |9.97                      |
|2019-01-01|9E_CR_0002|9E_AC_0002 |4            |4                |450                |1140.0          |690.0          |132.0          |14.95           |10.93                     |
|2019-01-01|9E_CR_0003|9E_AC_0003 |6            |6                |485                |1330.0  

In [0]:
crew_daily_summary.select(
    "TOTAL_DELAY_MIN", "EFFECTIVE_DELAY_MIN", "TOTAL_DUTY_HOURS"
).show(10, False)

+---------------+-------------------+----------------+
|TOTAL_DELAY_MIN|EFFECTIVE_DELAY_MIN|TOTAL_DUTY_HOURS|
+---------------+-------------------+----------------+
|0.0            |0.0                |2.57            |
|0.0            |0.0                |13.22           |
|53.0           |26.5               |4.52            |
|122.0          |61.0               |14.5            |
|0.0            |0.0                |12.37           |
|0.0            |0.0                |13.15           |
|0.0            |0.0                |6.97            |
|87.0           |43.5               |13.9            |
|0.0            |0.0                |4.53            |
|0.0            |0.0                |16.25           |
+---------------+-------------------+----------------+
only showing top 10 rows


In [0]:
crew_daily_summary.filter(col("CREW_ID") == "AA_CR_0028").select(
    "FL_DATE",
    "CREW_ID",
    "TOTAL_FLIGHTS",
    "DUTY_WINDOW_MIN",
    "TOTAL_DELAY_MIN",
    "EFFECTIVE_DELAY_MIN",
    "TOTAL_DUTY_HOURS",
).show(truncate=False)

+----------+----------+-------------+---------------+---------------+-------------------+----------------+
|FL_DATE   |CREW_ID   |TOTAL_FLIGHTS|DUTY_WINDOW_MIN|TOTAL_DELAY_MIN|EFFECTIVE_DELAY_MIN|TOTAL_DUTY_HOURS|
+----------+----------+-------------+---------------+---------------+-------------------+----------------+
|2019-02-02|AA_CR_0028|4            |645.0          |36.0           |18.0               |12.6            |
|2019-11-08|AA_CR_0028|3            |579.0          |71.0           |35.5               |12.08           |
|2021-11-13|AA_CR_0028|3            |591.0          |0.0            |0.0                |11.1            |
|2022-06-01|AA_CR_0028|2            |549.0          |0.0            |0.0                |10.4            |
|2019-05-09|AA_CR_0028|4            |603.0          |21.0           |10.5               |11.65           |
|2021-03-03|AA_CR_0028|1            |119.0          |0.0            |0.0                |3.23            |
|2021-04-19|AA_CR_0028|3            |

#### Validate Duty Window

In [0]:
from pyspark.sql.functions import *

crew_daily_summary.filter(col("DUTY_WINDOW_MIN") < 0).count()

0

In [0]:
crew_daily_summary.select(max("TOTAL_DUTY_HOURS").alias("MAX_DUTY_HOURS")).show()

+--------------+
|MAX_DUTY_HOURS|
+--------------+
|         67.07|
+--------------+



In [0]:
crew_simulation_df.filter(
    (col("CREW_ID") == "AA_CR_0025") & (col("FL_DATE") == "2023-05-15")
).select(
    "FL_NUMBER",
    "ORIGIN_AIRPORT_CODE",
    "DEST_AIRPORT_CODE",
    "TOTAL_DELAY",
    "FLIGHT_LEG_NUMBER",
).show(
    100, False
)

+---------+-------------------+-----------------+-----------+-----------------+
|FL_NUMBER|ORIGIN_AIRPORT_CODE|DEST_AIRPORT_CODE|TOTAL_DELAY|FLIGHT_LEG_NUMBER|
+---------+-------------------+-----------------+-----------+-----------------+
|2344.0   |ATL                |MIA              |3237.0     |1                |
|500.0    |PHX                |BOI              |0.0        |2                |
|801.0    |PHL                |CLE              |75.0       |3                |
|1894.0   |CLT                |RDU              |0.0        |4                |
+---------+-------------------+-----------------+-----------+-----------------+



In [0]:
crew_simulation_df.filter(
    (col("CREW_ID") == "AA_CR_0025") & (col("FL_DATE") == "2023-05-15")
).count()

4

In [0]:
crew_base_df.filter(
    (col("FL_NUMBER") == 2344)
    & (col("FL_DATE") == "2023-05-15")
    & (col("AIRLINE_CODE") == "AA")
).select("DEP_DELAY", "ARR_DELAY", "TOTAL_DELAY", "DOMINANT_DELAY_REASON").show(
    truncate=False
)

+---------+---------+-----------+---------------------+
|DEP_DELAY|ARR_DELAY|TOTAL_DELAY|DOMINANT_DELAY_REASON|
+---------+---------+-----------+---------------------+
|3221.0   |3237.0   |3237.0     |CARRIER              |
+---------+---------+-----------+---------------------+



#### Compute Normalization Values

In [0]:
from pyspark.sql.functions import *

# ==================================================
# Maximum Values for Normalization
# ==================================================

max_values = crew_daily_summary.agg(
    max("TOTAL_DUTY_HOURS").alias("MAX_DUTY"),
    max("TOTAL_FLIGHTS").alias("MAX_FLIGHTS"),
    max("TOTAL_DELAY_MIN").alias("MAX_DELAY"),
    max("AIRCRAFT_UTILIZATION_HOURS").alias("MAX_UTIL"),
).collect()[0]

MAX_DUTY = max_values["MAX_DUTY"]
MAX_FLIGHTS = max_values["MAX_FLIGHTS"]
MAX_DELAY = max_values["MAX_DELAY"]
MAX_UTIL = max_values["MAX_UTIL"]

print("=" * 100)
print("Normalization Values")
print("=" * 100)

print("Duty Hours :", MAX_DUTY)
print("Flights    :", MAX_FLIGHTS)
print("Delay      :", MAX_DELAY)
print("Utilization:", MAX_UTIL)

Normalization Values
Duty Hours : 67.07
Flights    : 7
Delay      : 3312.0
Utilization: 24.73


#### Calculate the Fatigue Score

In [0]:
crew_daily_summary = crew_daily_summary.withColumn(
    "FATIGUE_SCORE",
    round(
        (
            (least(col("TOTAL_DUTY_HOURS"), lit(MAX_DUTY)) / lit(MAX_DUTY)) * 45
            + (least(col("TOTAL_FLIGHTS"), lit(MAX_FLIGHTS)) / lit(MAX_FLIGHTS)) * 20
            + (least(col("TOTAL_DELAY_MIN"), lit(MAX_DELAY)) / lit(MAX_DELAY)) * 20
            + (least(col("AIRCRAFT_UTILIZATION_HOURS"), lit(MAX_UTIL)) / lit(MAX_UTIL))
            * 15
        ),
        2,
    ),
)

In [0]:
crew_daily_summary.show(6)

+----------+------------+----------+-----------+-------------+-----------------+-------------------+----------------+---------------+--------------------+-----------------------+------------------------+---------------+-------------------+-----------------+----------------+--------------+----------------+--------------------------+-------------+
|   FL_DATE|AIRLINE_CODE|   CREW_ID|AIRCRAFT_ID|TOTAL_FLIGHTS|TOTAL_FLIGHT_LEGS|FIRST_DEPARTURE_MIN|LAST_ARRIVAL_MIN|TOTAL_DELAY_MIN|AVG_DELAY_PER_FLIGHT|AVG_FLIGHT_DURATION_MIN|AIRCRAFT_UTILIZATION_MIN|DUTY_WINDOW_MIN|EFFECTIVE_DELAY_MIN|BRIEFING_TIME_MIN|DEBRIEF_TIME_MIN|TOTAL_DUTY_MIN|TOTAL_DUTY_HOURS|AIRCRAFT_UTILIZATION_HOURS|FATIGUE_SCORE|
+----------+------------+----------+-----------+-------------+-----------------+-------------------+----------------+---------------+--------------------+-----------------------+------------------------+---------------+-------------------+-----------------+----------------+--------------+---------------

#### Classify Fatigue Level

In [0]:
crew_daily_summary = crew_daily_summary.withColumn(
    "FATIGUE_LEVEL",
    when(col("FATIGUE_SCORE") < 15, "LOW")
    .when(col("FATIGUE_SCORE") < 30, "MODERATE")
    .when(col("FATIGUE_SCORE") < 45, "HIGH")
    .otherwise("CRITICAL"),
)

#### Calculate the Crew Health Index

In [0]:
crew_daily_summary = crew_daily_summary.withColumn(
    "CREW_HEALTH_INDEX", round(100 - col("FATIGUE_SCORE"), 2)
)
print("=" * 100)
print("Crew Health Index Created")
print("=" * 100)
crew_daily_summary.select("CREW_ID", "FATIGUE_SCORE", "CREW_HEALTH_INDEX").show(
    10, truncate=False
)

Crew Health Index Created
+----------+-------------+-----------------+
|CREW_ID   |FATIGUE_SCORE|CREW_HEALTH_INDEX|
+----------+-------------+-----------------+
|9E_CR_0015|5.84         |94.16            |
|9E_CR_0007|29.72        |70.28            |
|9E_CR_0014|8.11         |91.89            |
|9E_CR_0012|24.19        |75.81            |
|9E_CR_0005|25.66        |74.34            |
|9E_CR_0010|26.84        |73.16            |
|9E_CR_0007|13.93        |86.07            |
|9E_CR_0001|33.18        |66.82            |
|9E_CR_0009|11.06        |88.94            |
|9E_CR_0007|31.3         |68.7             |
+----------+-------------+-----------------+
only showing top 10 rows


#### Generate Operational Recommendations

In [0]:
crew_daily_summary = crew_daily_summary.withColumn(
    "OPERATIONAL_RECOMMENDATION",
    when(col("FATIGUE_LEVEL") == "LOW", "Continue Operations")
    .when(col("FATIGUE_LEVEL") == "MODERATE", "Schedule Rest")
    .when(col("FATIGUE_LEVEL") == "HIGH", "Replace After Current Flight")
    .otherwise("Immediate Crew Replacement"),
)
print("=" * 100)
print("Operational Recommendations")
print("=" * 100)
crew_daily_summary.groupBy("OPERATIONAL_RECOMMENDATION").count().show(truncate=False)

Operational Recommendations
+----------------------------+------+
|OPERATIONAL_RECOMMENDATION  |count |
+----------------------------+------+
|Immediate Crew Replacement  |405   |
|Schedule Rest               |506497|
|Replace After Current Flight|43459 |
|Continue Operations         |197670|
+----------------------------+------+



In [0]:
crew_daily_summary.select(
    "FL_DATE",
    "AIRLINE_CODE",
    "CREW_ID",
    "TOTAL_FLIGHTS",
    "TOTAL_DUTY_HOURS",
    "TOTAL_DELAY_MIN",
    "FATIGUE_SCORE",
    "FATIGUE_LEVEL",
    "CREW_HEALTH_INDEX",
    "OPERATIONAL_RECOMMENDATION",
).orderBy(desc("FATIGUE_SCORE")).show(30, truncate=False)

+----------+------------+----------+-------------+----------------+---------------+-------------+-------------+-----------------+----------------------------+
|FL_DATE   |AIRLINE_CODE|CREW_ID   |TOTAL_FLIGHTS|TOTAL_DUTY_HOURS|TOTAL_DELAY_MIN|FATIGUE_SCORE|FATIGUE_LEVEL|CREW_HEALTH_INDEX|OPERATIONAL_RECOMMENDATION  |
+----------+------------+----------+-------------+----------------+---------------+-------------+-------------+-----------------+----------------------------+
|2023-05-15|AA          |AA_CR_0025|4            |67.07           |3312.0         |82.9         |CRITICAL     |17.1             |Immediate Crew Replacement  |
|2021-06-16|AA          |AA_CR_0030|3            |66.38           |3070.0         |78.19        |CRITICAL     |21.81            |Immediate Crew Replacement  |
|2019-12-29|OO          |OO_CR_0013|3            |62.48           |3107.0         |75.18        |CRITICAL     |24.82            |Immediate Crew Replacement  |
|2022-08-27|AA          |AA_CR_0009|4         

#### Detect Duty Time Breaches

In [0]:
from pyspark.sql.functions import *

DUTY_LIMIT = 11
crew_daily_summary = (
    crew_daily_summary.withColumn(
        "BREACH_HOURS",
        round(greatest(col("TOTAL_DUTY_HOURS") - lit(DUTY_LIMIT), lit(0)), 2),
    )
    .withColumn("DUTY_BREACH", when(col("BREACH_HOURS") > 0, "YES").otherwise("NO"))
    .withColumn(
        "DUTY_STATUS",
        when(col("TOTAL_DUTY_HOURS") < 8, "SAFE")
        .when(col("TOTAL_DUTY_HOURS") < 11, "MODERATE")
        .when(col("TOTAL_DUTY_HOURS") < 13, "HIGH")
        .otherwise("CRITICAL"),
    )
)

In [0]:
crew_daily_summary.select(
    "CREW_ID", "TOTAL_DUTY_HOURS", "BREACH_HOURS", "DUTY_BREACH", "DUTY_STATUS"
).show(20, truncate=False)

+----------+----------------+------------+-----------+-----------+
|CREW_ID   |TOTAL_DUTY_HOURS|BREACH_HOURS|DUTY_BREACH|DUTY_STATUS|
+----------+----------------+------------+-----------+-----------+
|9E_CR_0015|2.57            |0.0         |NO         |SAFE       |
|9E_CR_0007|13.22           |2.22        |YES        |CRITICAL   |
|9E_CR_0014|4.52            |0.0         |NO         |SAFE       |
|9E_CR_0012|14.5            |3.5         |YES        |CRITICAL   |
|9E_CR_0005|12.37           |1.37        |YES        |HIGH       |
|9E_CR_0010|13.15           |2.15        |YES        |CRITICAL   |
|9E_CR_0007|6.97            |0.0         |NO         |SAFE       |
|9E_CR_0001|13.9            |2.9         |YES        |CRITICAL   |
|9E_CR_0009|4.53            |0.0         |NO         |SAFE       |
|9E_CR_0007|16.25           |5.25        |YES        |CRITICAL   |
|9E_CR_0004|14.27           |3.27        |YES        |CRITICAL   |
|9E_CR_0009|12.7            |1.7         |YES        |HIGH    

#### Assemble the Gold Crew Intelligence Table

In [0]:
gold_crew_intelligence = crew_daily_summary.select(
    # =====================================================
    # Crew Information
    # =====================================================
    "FL_DATE",
    "AIRLINE_CODE",
    "CREW_ID",
    "AIRCRAFT_ID",
    # =====================================================
    # Operational KPIs
    # =====================================================
    "TOTAL_FLIGHTS",
    "TOTAL_FLIGHT_LEGS",
    "TOTAL_DUTY_HOURS",
    "TOTAL_DELAY_MIN",
    "AVG_DELAY_PER_FLIGHT",
    "AVG_FLIGHT_DURATION_MIN",
    "AIRCRAFT_UTILIZATION_HOURS",
    # =====================================================
    # Crew Risk KPIs
    # =====================================================
    "DUTY_STATUS",
    "DUTY_BREACH",
    "BREACH_HOURS",
    "FATIGUE_SCORE",
    "FATIGUE_LEVEL",
    "CREW_HEALTH_INDEX",
    # =====================================================
    # Recommendation
    # =====================================================
    "OPERATIONAL_RECOMMENDATION",
)

#### Write the Gold Crew Intelligence Table

In [0]:
gold_crew_intelligence.write.mode("overwrite").saveAsTable(
    "aviation_gold.crew_intelligence"
)